# Exploring Splice Site Predictions with AgenticSpliceAI Lab
## SERPINA1 (COPD), UNC13A & STMN2 (ALS)

This notebook walks through a bioinformatics investigation using the AgenticSpliceAI Lab web UI API. We explore three clinically important genes:

- **SERPINA1** — encodes alpha-1 antitrypsin (AAT). Deficiency causes AATD, the most common genetic cause of COPD in young adults (~1 in 2,500 of European descent).
- **UNC13A** — synaptic vesicle priming protein. An intronic SNP creates a cryptic splice site that is unmasked when TDP-43 is depleted in ALS/FTD.
- **STMN2** — stathmin-2, essential for axonal regeneration. TDP-43 loss leads to cryptic exon inclusion, producing truncated non-functional protein — a hallmark of ALS motor neurons.

We'll use the Lab's REST API to browse genes, run predictions, analyze errors, and compare models.

### Prerequisites
- AgenticSpliceAI Lab server running on `localhost:8005`
- Start with: `conda run -n agentic-spliceai python -m server.bio.app`
- Python packages: `requests`, `plotly`, `pandas`, `numpy`

### Background
See [`supplements/01_serpina1_biology.md`](supplements/01_serpina1_biology.md) for detailed biology.

In [ ]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import requests

# API base URL — server must be running
BASE_URL = "http://localhost:8005"

# Color palette (matches the web UI exactly)
COLORS = {
    "donor": "#3572a5",       # Blue
    "acceptor": "#e74c3c",    # Red
    "tp": "#27ae60",          # Green
    "fp": "#e74c3c",          # Red
    "fn": "#f39c12",          # Amber
    "navy": "#1e3a5f",        # Dark navy
    "threshold": "#cccccc",   # Light gray
}


def api_get(endpoint, params=None):
    """GET request to the AgenticSpliceAI Lab API."""
    resp = requests.get(f"{BASE_URL}{endpoint}", params=params, timeout=120)
    resp.raise_for_status()
    return resp.json()


# Test connection
try:
    models = api_get("/api/models")
    print(f"Server is running. Available models: {[m['name'] for m in models]}")
except requests.ConnectionError:
    print("ERROR: Cannot connect to server at localhost:8005")
    print("Start it with: conda run -n agentic-spliceai python -m server.bio.app")

## 1. Discover Available Models

Each model uses a specific genome build and annotation source.

In [ ]:
models = api_get("/api/models")
models_df = pd.DataFrame(models)
display(models_df)

# We'll use openspliceai (GRCh38/MANE) as the primary model
MODEL = "openspliceai"
print(f"\nUsing model: {MODEL}")

**Model fields:**
- `name` — model identifier (e.g., `openspliceai`, `spliceai`)
- `build` — genome assembly (`GRCh38` = hg38, `GRCh37` = hg19)
- `annotation_source` — gene annotations (`mane` = MANE Select, `ensembl` = full Ensembl)

We use `openspliceai` with GRCh38/MANE — the most current combination.

## 2. Browse the Gene Catalog

The Gene Browser provides a searchable, paginated catalog of all protein-coding genes.

In [ ]:
stats = api_get("/api/genes/stats", params={"model": MODEL})
print(f"Model: {stats['model']} ({stats['build']}, {stats['annotation_source']})")
print(f"Total protein-coding genes: {stats['total_genes']:,}")
print(f"\nTop 10 chromosomes by gene count:")

per_chr = sorted(stats['per_chromosome'].items(), key=lambda x: x[1], reverse=True)
for chrom, count in per_chr[:10]:
    print(f"  {chrom}: {count:,}")

In [ ]:
# Chromosome distribution bar chart (canonical chromosomes only)
import re

CANONICAL = {f"chr{i}" for i in range(1, 23)} | {"chrX", "chrY"}

def chr_sort_key(name):
    """Natural sort: chr1, chr2, ..., chr22, chrX, chrY."""
    m = re.match(r"chr(\d+)", name)
    if m:
        return (0, int(m.group(1)))
    return (1, name)  # chrX, chrY after numerics

canonical = {k: v for k, v in stats['per_chromosome'].items() if k in CANONICAL}
chroms_sorted = sorted(canonical.items(), key=lambda x: chr_sort_key(x[0]))
chr_names = [c[0] for c in chroms_sorted]
chr_counts = [c[1] for c in chroms_sorted]

fig = go.Figure(data=[
    go.Bar(x=chr_names, y=chr_counts, marker_color=COLORS["navy"])
])
fig.update_layout(
    title="Protein-Coding Genes per Chromosome",
    xaxis_title="Chromosome",
    yaxis_title="Gene Count",
    template="plotly_white",
    height=350,
)

# Note how many genes are on patch/alt contigs
n_alt = sum(v for k, v in stats['per_chromosome'].items() if k not in CANONICAL)
if n_alt > 0:
    print(f"({n_alt} additional genes on patch/alt contigs not shown)")

fig.show()

In [ ]:
# Search for our target genes
target_genes = ["SERPINA1", "UNC13A", "STMN2"]

for gene in target_genes:
    result = api_get("/api/genes", params={"model": MODEL, "search": gene, "per_page": 5})
    genes = result['genes']
    if genes:
        g = genes[0]
        print(f"{g['gene_name']:>10}  {g['chrom']}:{g['start']:>10,}-{g['end']:>10,}  "
              f"({g['length']:,} bp)  {g['n_splice_sites']} splice sites  {g['description'][:60]}")
    else:
        print(f"{gene:>10}  — not found in {MODEL}")

### Clinical Context

**SERPINA1** encodes alpha-1 antitrypsin (AAT), the primary protease inhibitor protecting lungs from neutrophil elastase. Deficiency causes AATD — the most common genetic cause of COPD in young adults. Splice site variants can cause null alleles (complete AAT loss), which are often more severe than the common Z/S missense alleles.

**UNC13A** is a synaptic vesicle priming protein. A common intronic SNP (rs12608932) creates a cryptic splice site that becomes active when TDP-43 is depleted in ALS/FTD — making it one of the most important genetic modifiers of disease.

**STMN2** (stathmin-2) is essential for axonal regeneration. TDP-43 loss leads to cryptic exon inclusion between exons 1 and 2, producing a truncated protein. Reduced STMN2 is a hallmark of ALS motor neurons.

For all three genes, accurate splice site prediction is directly relevant to understanding disease mechanisms.

## 3. Genome View: Splice Site Predictions

The prediction API runs the full pipeline for a single gene:
1. Extract gene sequence and ground truth annotations
2. Run the neural network to predict splice probabilities at every position
3. Classify predictions as TP (true positive), FP (false positive), or FN (false negative)

Let's start with **SERPINA1**.

In [ ]:
THRESHOLD = 0.5


def run_prediction(gene_name, model=MODEL, threshold=THRESHOLD):
    """Run prediction and print summary."""
    print(f"Running prediction for {gene_name} (model={model}, threshold={threshold})...")
    t0 = time.time()
    data = api_get(f"/api/genome/{gene_name}/predict", params={
        "model": model, "threshold": threshold,
    })
    elapsed = time.time() - t0

    print(f"  Completed in {elapsed:.1f}s")
    print(f"  {data['gene_name']} ({data['gene_id']})")
    print(f"  {data['chrom']}:{data['gene_start']:,}-{data['gene_end']:,} ({data['strand']})")
    print(f"  {data['total_positions']:,} positions, downsample={data['downsample_factor']}")
    print(f"  TP={data['n_tp']}  FP={data['n_fp']}  FN={data['n_fn']}")

    tp, fn = data['n_tp'], data['n_fn']
    fp = data['n_fp']
    if tp + fn > 0:
        recall = tp / (tp + fn)
        print(f"  Recall: {recall:.1%}", end="")
    if tp + fp > 0:
        precision = tp / (tp + fp)
        print(f"  Precision: {precision:.1%}", end="")
    if tp + fn > 0 and tp + fp > 0 and (precision + recall) > 0:
        f1 = 2 * precision * recall / (precision + recall)
        print(f"  F1: {f1:.1%}", end="")
    print()
    return data


serpina1 = run_prediction("SERPINA1")

In [ ]:
def plot_genome_view(data, height=750):
    """Three-track genome view replicating the web UI."""
    fig = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.06,
        row_heights=[0.30, 0.30, 0.40],
        subplot_titles=("Donor Probability", "Acceptor Probability",
                        "Classification (TP / FP / FN)"),
    )

    # Track 1: Donor probability
    fig.add_trace(go.Scatter(
        x=data['positions'], y=data['donor_prob'],
        mode='lines', name='Donor Prob',
        line=dict(color=COLORS['donor'], width=1.2),
        hovertemplate='Pos: %{x:,}<br>Prob: %{y:.4f}<extra>Donor</extra>',
    ), row=1, col=1)
    fig.add_hline(y=data['threshold'], line_dash="dot",
                  line_color=COLORS['threshold'], row=1, col=1)

    # Track 2: Acceptor probability
    fig.add_trace(go.Scatter(
        x=data['positions'], y=data['acceptor_prob'],
        mode='lines', name='Acceptor Prob',
        line=dict(color=COLORS['acceptor'], width=1.2),
        hovertemplate='Pos: %{x:,}<br>Prob: %{y:.4f}<extra>Acceptor</extra>',
    ), row=2, col=1)
    fig.add_hline(y=data['threshold'], line_dash="dot",
                  line_color=COLORS['threshold'], row=2, col=1)

    # Track 3: Classification markers
    for pred_type, color, symbol, label in [
        ('TP', COLORS['tp'], 'circle', 'True Positive'),
        ('FP', COLORS['fp'], 'triangle-up', 'False Positive'),
        ('FN', COLORS['fn'], 'x', 'False Negative'),
    ]:
        filtered = [m for m in data['markers'] if m['pred_type'] == pred_type]
        if filtered:
            fig.add_trace(go.Scatter(
                x=[m['position'] for m in filtered],
                y=[m['donor_score'] if m['site_type'] == 'donor'
                   else m['acceptor_score'] for m in filtered],
                mode='markers', name=f"{label} ({len(filtered)})",
                marker=dict(color=color, size=9, symbol=symbol),
                customdata=[
                    f"{m['site_type']} | D={m['donor_score']:.3f} A={m['acceptor_score']:.3f}"
                    for m in filtered
                ],
                hovertemplate=(
                    'Pos: %{x:,}<br>Score: %{y:.4f}<br>'
                    '%{customdata}<extra>' + pred_type + '</extra>'
                ),
            ), row=3, col=1)

    # Ground truth vertical lines
    for pos, stype in zip(data['gt_positions'], data['gt_site_types']):
        color = COLORS['donor'] if stype == 'donor' else COLORS['acceptor']
        fig.add_vline(x=pos, line_color=color, opacity=0.15,
                      line_width=1, row=3, col=1)

    fig.update_yaxes(range=[0, 1.05], row=1, col=1)
    fig.update_yaxes(range=[0, 1.05], row=2, col=1)
    fig.update_yaxes(range=[0, 1.1], row=3, col=1)
    fig.update_xaxes(title_text=f"Genomic Position ({data['chrom']})", row=3, col=1)

    fig.update_layout(
        height=height,
        template="plotly_white",
        title_text=(
            f"Genome View: {data['gene_name']} "
            f"({data['chrom']}:{data['gene_start']:,}-{data['gene_end']:,})"
        ),
        showlegend=True,
        legend=dict(orientation='h', y=-0.08, x=0.5, xanchor='center'),
    )
    fig.show()


plot_genome_view(serpina1)

### Reading the Genome View

- **Donor Probability (top)**: Blue line — model confidence that each position is a donor splice site (5' intron boundary, typically GT). Peaks above the dashed threshold are predicted donors.
- **Acceptor Probability (middle)**: Red line — acceptor splice site probability (3' intron boundary, typically AG).
- **Classification (bottom)**: Evaluation against ground truth annotations:
  - Green circles (**TP**) — correctly identified splice sites
  - Red triangles (**FP**) — predicted where none is annotated
  - Amber X marks (**FN**) — missed annotated splice sites
  - Faint vertical lines — ground truth positions (blue=donor, red=acceptor)

Hover over any point for exact coordinates and scores.

### Now let's run predictions for the ALS genes

In [ ]:
print("=" * 60)
unc13a = run_prediction("UNC13A")
print()
print("=" * 60)
stmn2 = run_prediction("STMN2")

In [ ]:
plot_genome_view(unc13a)

In [ ]:
plot_genome_view(stmn2)

## 4. Analyzing Prediction Errors: FP and FN Sites

Understanding *why* the model makes errors is the most scientifically interesting part:
- **False positives** may indicate cryptic splice sites or alternative splicing not captured in annotations
- **False negatives** may indicate weak splice signals hard to detect from sequence alone

Let's analyze all three genes.

In [ ]:
def analyze_errors(data):
    """Analyze FP/FN markers for a prediction result."""
    markers_df = pd.DataFrame(data['markers'])
    gene = data['gene_name']
    chrom = data['chrom']

    print(f"{'=' * 50}")
    print(f"{gene} — {len(markers_df)} classified markers")
    print(f"{'=' * 50}")

    if len(markers_df) == 0:
        print("  No markers (no splice sites predicted or annotated)")
        return markers_df

    # Breakdown
    print(markers_df.groupby(['pred_type', 'site_type']).size()
          .unstack(fill_value=0).to_string())

    # Add max_score column
    markers_df['max_score'] = markers_df.apply(
        lambda r: r['donor_score'] if r['site_type'] == 'donor'
        else r['acceptor_score'], axis=1,
    )

    # FP analysis
    fp = markers_df[markers_df['pred_type'] == 'FP']
    if len(fp) > 0:
        print(f"\n  False Positives ({len(fp)}):")
        for _, row in fp.iterrows():
            print(f"    {chrom}:{row['position']:>10,}  {row['site_type']:>8}  "
                  f"score={row['max_score']:.4f}")

    # FN analysis
    fn = markers_df[markers_df['pred_type'] == 'FN']
    if len(fn) > 0:
        print(f"\n  False Negatives ({len(fn)}):")
        for _, row in fn.iterrows():
            deficit = THRESHOLD - row['max_score']
            print(f"    {chrom}:{row['position']:>10,}  {row['site_type']:>8}  "
                  f"score={row['max_score']:.4f}  deficit={deficit:.4f}")

    if len(fp) == 0 and len(fn) == 0:
        print("\n  Perfect prediction — no errors!")

    return markers_df


serpina1_markers = analyze_errors(serpina1)
print()
unc13a_markers = analyze_errors(unc13a)
print()
stmn2_markers = analyze_errors(stmn2)

In [ ]:
# Score distribution comparison across all three genes
all_markers = []
for data, name in [(serpina1, 'SERPINA1'), (unc13a, 'UNC13A'), (stmn2, 'STMN2')]:
    for m in data['markers']:
        score = m['donor_score'] if m['site_type'] == 'donor' else m['acceptor_score']
        all_markers.append({
            'gene': name,
            'pred_type': m['pred_type'],
            'score': score,
            'site_type': m['site_type'],
        })

all_df = pd.DataFrame(all_markers)

fig = go.Figure()
for pred_type, color, label in [
    ('TP', COLORS['tp'], 'True Positive'),
    ('FP', COLORS['fp'], 'False Positive'),
    ('FN', COLORS['fn'], 'False Negative'),
]:
    subset = all_df[all_df['pred_type'] == pred_type]
    if len(subset) == 0:
        continue
    fig.add_trace(go.Box(
        y=subset['score'],
        name=f"{label} (n={len(subset)})",
        marker_color=color,
        boxpoints='all', jitter=0.3, pointpos=-1.5,
    ))

fig.update_layout(
    title="Score Distribution: TP vs FP vs FN (All Genes)",
    yaxis_title="Prediction Score",
    template="plotly_white",
    height=400,
    showlegend=False,
)
fig.show()

In [ ]:
# FP proximity to nearest ground truth site
for data, name in [(serpina1, 'SERPINA1'), (unc13a, 'UNC13A'), (stmn2, 'STMN2')]:
    fp_markers = [m for m in data['markers'] if m['pred_type'] == 'FP']
    if not fp_markers:
        continue

    gt_pos = np.array(data['gt_positions'])
    print(f"{name} — FP distance to nearest ground truth:")
    for m in fp_markers:
        dist = int(np.min(np.abs(gt_pos - m['position']))) if len(gt_pos) > 0 else -1
        score = m['donor_score'] if m['site_type'] == 'donor' else m['acceptor_score']
        print(f"  pos={m['position']:,}  nearest_gt={dist:,}bp  "
              f"type={m['site_type']}  score={score:.4f}")
    print()

### Interpreting the Errors

**False Positives** (model predicted, not annotated):
- High-scoring FPs near exon boundaries may indicate cryptic splice sites or alternative isoforms not in the MANE Select annotation
- For UNC13A specifically, FPs in introns could correspond to the TDP-43-dependent cryptic exon
- In clinical settings, FPs near disease-associated regions deserve manual review

**False Negatives** (annotated, model missed):
- FNs with scores near the threshold may be recoverable by lowering it
- FNs with very low scores (<0.1) suggest the model cannot detect those sites from sequence alone
- For SERPINA1, missed splice sites could mean a pathogenic splicing variant at that position would go undetected

**Clinical takeaway**: In diagnostics, FNs are more dangerous than FPs — a missed splice site variant could mean a missed diagnosis.

## 5. Threshold Sensitivity Analysis

The classification threshold determines which predictions are called as splice sites. Lowering it recovers more true sites (higher recall) but increases false positives (lower precision).

Since the API caches raw predictions, only the cheap evaluation step re-runs on threshold change.

In [ ]:
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]


def sweep_thresholds(gene_name, thresholds=thresholds):
    """Sweep thresholds and collect TP/FP/FN/metrics."""
    rows = []
    for t in thresholds:
        pred = api_get(f"/api/genome/{gene_name}/predict", params={
            "model": MODEL, "threshold": t,
        })
        tp, fp, fn = pred['n_tp'], pred['n_fp'], pred['n_fn']
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        f1 = (2 * precision * recall / (precision + recall)
              if (precision + recall) > 0 else 0)
        rows.append({
            'threshold': t, 'TP': tp, 'FP': fp, 'FN': fn,
            'recall': recall, 'precision': precision, 'f1': f1,
        })
    return pd.DataFrame(rows)


print("Sweeping thresholds...")
t0 = time.time()
serpina1_thresh = sweep_thresholds("SERPINA1")
unc13a_thresh = sweep_thresholds("UNC13A")
stmn2_thresh = sweep_thresholds("STMN2")
print(f"Done in {time.time() - t0:.1f}s (cached predictions, evaluation only)")

display(serpina1_thresh)

In [ ]:
# Precision / Recall / F1 vs Threshold for all three genes
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('SERPINA1', 'UNC13A', 'STMN2'),
    shared_yaxes=True,
)

for col, (df, name) in enumerate([
    (serpina1_thresh, 'SERPINA1'),
    (unc13a_thresh, 'UNC13A'),
    (stmn2_thresh, 'STMN2'),
], start=1):
    show_legend = col == 1
    fig.add_trace(go.Scatter(
        x=df['threshold'], y=df['recall'],
        mode='lines+markers', name='Recall',
        line=dict(color=COLORS['donor'], width=2),
        marker=dict(size=6), showlegend=show_legend,
    ), row=1, col=col)
    fig.add_trace(go.Scatter(
        x=df['threshold'], y=df['precision'],
        mode='lines+markers', name='Precision',
        line=dict(color=COLORS['acceptor'], width=2),
        marker=dict(size=6), showlegend=show_legend,
    ), row=1, col=col)
    fig.add_trace(go.Scatter(
        x=df['threshold'], y=df['f1'],
        mode='lines+markers', name='F1',
        line=dict(color=COLORS['navy'], width=2, dash='dash'),
        marker=dict(size=6), showlegend=show_legend,
    ), row=1, col=col)
    fig.add_vline(x=0.5, line_dash="dot", line_color="gray",
                  row=1, col=col)

fig.update_yaxes(range=[0, 1.05], row=1, col=1)
fig.update_layout(
    title="Threshold Sensitivity: Precision / Recall / F1",
    height=350,
    template="plotly_white",
    legend=dict(orientation='h', y=-0.15, x=0.5, xanchor='center'),
)
for col in range(1, 4):
    fig.update_xaxes(title_text="Threshold", row=1, col=col)
fig.show()

In [ ]:
# TP/FP/FN count breakdown across thresholds (SERPINA1 as example)
fig = go.Figure()
for metric, color, label in [
    ('TP', COLORS['tp'], 'True Positives'),
    ('FP', COLORS['fp'], 'False Positives'),
    ('FN', COLORS['fn'], 'False Negatives'),
]:
    fig.add_trace(go.Scatter(
        x=serpina1_thresh['threshold'], y=serpina1_thresh[metric],
        mode='lines+markers', name=label,
        line=dict(color=color, width=2),
        fill='tozeroy', fillcolor=color.replace(')', ', 0.1)').replace('rgb', 'rgba')
        if color.startswith('rgb') else None,
    ))

fig.add_vline(x=0.5, line_dash="dot", line_color="gray",
              annotation_text="Default")
fig.update_layout(
    title="SERPINA1: Error Counts vs Threshold",
    xaxis_title="Threshold",
    yaxis_title="Count",
    template="plotly_white",
    height=350,
    legend=dict(orientation='h', y=-0.15, x=0.5, xanchor='center'),
)
fig.show()

### Choosing a Threshold

**For diagnostic screening** (analyzing a patient's variants for splice effects):
- Lower threshold (0.2–0.3) — better to flag too many sites than miss one
- FPs can be filtered downstream by expert review or RNA-seq validation

**For research annotation** (building a splice site catalog):
- Default threshold (0.5) or higher — minimizes noise

The optimal threshold depends on the cost asymmetry between FPs and FNs in your application.

## 6. Model Comparison: Metrics Dashboard

The Metrics Dashboard compares evaluation runs across models. Let's pull available runs.

In [ ]:
runs = api_get("/api/metrics/runs")
if runs:
    runs_df = pd.DataFrame(runs)
    display(runs_df[['run_id', 'model', 'build', 'annotation_source',
                      'n_genes', 'threshold']])
else:
    print("No evaluation runs found.")
    print("Run: python examples/base_layer/03_prediction_with_evaluation.py")

In [ ]:
if len(runs) >= 2:
    run_ids = [r['run_id'] for r in runs[:2]]
    comparison = api_get("/api/metrics/compare", params={"runs": ",".join(run_ids)})

    # Build comparison table
    rows = []
    for run_id, metrics_data in comparison.items():
        m = metrics_data['metrics']['canonical']['overall']
        meta = metrics_data['metadata']
        rows.append({
            'Model': meta['model'], 'Build': meta['build'],
            'TP': m['tp'], 'FP': m['fp'], 'FN': m['fn'],
            'Recall': f"{m['recall']:.1%}",
            'Precision': f"{m['precision']:.1%}",
            'F1': f"{m['f1']:.1%}",
        })
    display(pd.DataFrame(rows))

    # Side-by-side F1 comparison
    fig = go.Figure()
    bar_colors = [COLORS['navy'], COLORS['donor']]
    for i, (run_id, metrics_data) in enumerate(comparison.items()):
        m = metrics_data['metrics']['canonical']
        meta = metrics_data['metadata']
        label = f"{meta['model']} ({meta['build']})"
        fig.add_trace(go.Bar(
            x=['Donor F1', 'Acceptor F1', 'Overall F1'],
            y=[m['donor']['f1'] * 100, m['acceptor']['f1'] * 100,
               m['overall']['f1'] * 100],
            name=label, marker_color=bar_colors[i % len(bar_colors)],
            text=[f"{m['donor']['f1']*100:.1f}%", f"{m['acceptor']['f1']*100:.1f}%",
                  f"{m['overall']['f1']*100:.1f}%"],
            textposition='outside',
        ))
    fig.update_layout(
        title="Model Comparison: F1 Score by Site Type",
        yaxis=dict(title='F1 Score (%)', range=[0, 105]),
        barmode='group', template="plotly_white", height=400,
        legend=dict(orientation='h', y=-0.15, x=0.5, xanchor='center'),
    )
    fig.show()

elif len(runs) == 1:
    run_id = runs[0]['run_id']
    metrics_data = api_get(f"/api/metrics/{run_id}")
    m = metrics_data['metrics']['canonical']
    print(f"Single run: {runs[0]['model']} ({runs[0]['build']})")
    print(f"  Donor    — R={m['donor']['recall']:.1%}  P={m['donor']['precision']:.1%}  F1={m['donor']['f1']:.1%}")
    print(f"  Acceptor — R={m['acceptor']['recall']:.1%}  P={m['acceptor']['precision']:.1%}  F1={m['acceptor']['f1']:.1%}")
    print(f"  Overall  — R={m['overall']['recall']:.1%}  P={m['overall']['precision']:.1%}  F1={m['overall']['f1']:.1%}")
    print(f"\nRun evaluation with a second model to enable comparison.")
else:
    print("No evaluation runs available for comparison.")

## 7. Ground Truth Splice Sites

Let's examine the annotated splice sites to understand each gene's exon-intron structure.

In [ ]:
def plot_gene_map(data):
    """Minimal gene map showing splice site positions."""
    gt_df = pd.DataFrame({
        'position': data['gt_positions'],
        'site_type': data['gt_site_types'],
    })
    n_donors = (gt_df['site_type'] == 'donor').sum()
    n_acceptors = (gt_df['site_type'] == 'acceptor').sum()
    print(f"{data['gene_name']}: {len(gt_df)} splice sites "
          f"({n_donors} donor, {n_acceptors} acceptor) "
          f"=> ~{n_donors + 1} exons")

    fig = go.Figure()

    # Gene body line
    fig.add_shape(
        type='line',
        x0=data['gene_start'], x1=data['gene_end'],
        y0=0.5, y1=0.5,
        line=dict(color='#333', width=2),
    )

    donors = gt_df[gt_df['site_type'] == 'donor']['position'].tolist()
    acceptors = gt_df[gt_df['site_type'] == 'acceptor']['position'].tolist()

    fig.add_trace(go.Scatter(
        x=donors, y=[0.5] * len(donors),
        mode='markers', name="Donor (5')",
        marker=dict(color=COLORS['donor'], size=12, symbol='triangle-down'),
    ))
    fig.add_trace(go.Scatter(
        x=acceptors, y=[0.5] * len(acceptors),
        mode='markers', name="Acceptor (3')",
        marker=dict(color=COLORS['acceptor'], size=12, symbol='triangle-up'),
    ))

    fig.update_layout(
        title=f"{data['gene_name']} Splice Site Map ({data['chrom']})",
        xaxis_title="Genomic Position",
        yaxis=dict(visible=False, range=[0, 1]),
        template="plotly_white",
        height=180,
        legend=dict(orientation='h', y=-0.35, x=0.5, xanchor='center'),
        margin=dict(t=40, b=60),
    )
    fig.show()


plot_gene_map(serpina1)
plot_gene_map(unc13a)
plot_gene_map(stmn2)

### Gene Structure Notes

**SERPINA1**: 5 exons on chromosome 14. The reactive center loop (exon 5) is where most pathogenic missense variants occur. Splice site variants anywhere can cause null alleles — often more severe than Z/S missense alleles.

**UNC13A**: A large gene (~85 kb) on chromosome 19 with many exons. The critical cryptic exon between exons 20–21 is created by the rs12608932 variant and activated under TDP-43 depletion.

**STMN2**: Relatively compact gene on chromosome 8. The TDP-43-dependent cryptic exon between exons 1–2 is a hallmark of ALS and a therapeutic target.

## 8. Summary and Future Directions

### What We Explored

Using the AgenticSpliceAI Lab API, we:

1. **Discovered available models** and their genome builds
2. **Browsed ~19,000 protein-coding genes** and found our targets
3. **Generated splice site predictions** for SERPINA1, UNC13A, and STMN2
4. **Visualized three-track genome views** (donor/acceptor probabilities + classification)
5. **Analyzed prediction errors** — FP/FN sites with clinical context
6. **Explored threshold sensitivity** — precision/recall tradeoffs
7. **Compared model performance** across evaluation runs
8. **Mapped exon-intron structure** for all three genes

### Clinical Relevance

- **COPD/AATD**: Accurate splice site prediction for SERPINA1 can help identify pathogenic splicing variants that cause null alleles
- **ALS**: Detecting cryptic splice sites in UNC13A and STMN2 is directly relevant to understanding TDP-43-mediated neurodegeneration
- **Threshold choice**: Diagnostic applications should favor recall (lower threshold); research applications can favor precision

### Future Directions

- **Performance plots**: PR curves, per-gene error distributions, score histograms at FP/FN sites
- **Auto-generated reports**: Clinical-grade splice analysis reports for any gene
- **Agentic workflows**: LLM-powered analysis that fetches gene function, cross-references pathogenic variants, and explains clinical significance
- **Variant impact**: Predict how specific mutations affect splicing (e.g., SERPINA1 Z allele, UNC13A rs12608932)

In [ ]:
print("Session Information")
print("=" * 50)
print(f"Genes analyzed: SERPINA1, UNC13A, STMN2")
print(f"Model: {MODEL}")
print(f"Default threshold: {THRESHOLD}")
print(f"Server: {BASE_URL}")
print(f"\nAPI endpoints used:")
print(f"  GET /api/models")
print(f"  GET /api/genes/stats")
print(f"  GET /api/genes?search=...")
print(f"  GET /api/genome/<gene>/predict")
print(f"  GET /api/metrics/runs")
print(f"  GET /api/metrics/compare")
print(f"\nTo explore in the browser:")
print(f"  Gene Browser: {BASE_URL}/")
print(f"  SERPINA1:     {BASE_URL}/genome/SERPINA1")
print(f"  UNC13A:       {BASE_URL}/genome/UNC13A")
print(f"  STMN2:        {BASE_URL}/genome/STMN2")
print(f"  Metrics:      {BASE_URL}/metrics")